# 37 · 缓存与延迟优化

> RAG 慢在哪？检索前的 embedding，生成时的长 prompt 与 LLM。缓存是性价比最高的提速手段。

**本文件覆盖知识点**：Cache / Semantic Cache / Latency / Response Time / TTFT / Throughput / Cold Start / 优化清单

## 1. 延迟去哪了？

```text
总耗时 ≈ 请求路由 + 问题Embedding + 向量检索 + 重排 + LLM生成
                                                      └──── 大头，TTFT/首字与生成长度成正比
```

| 术语 | 含义 |
|------|------|
| **TTFT** | 首字返回时间（首 token 延迟） |
| **Throughput** | 每秒处理的请求/词元数（吞吐） |
| **Cold Start** | 冷启动：索引首次加载/进程首次拉起时的额外耗时 |

> 体感延迟 = TTFT + 生成时长；聊天机器人尤其看重 TTFT。

In [ ]:
# ===== 本课共用：真实检索底座 =====
# 真语料(data/) → 真切分 → 真向量(text-embedding-v3) → 真索引(FAISS + BM25)
# → 真重排(qwen3-rerank) → 真生成(qwen-plus)。各课在这个底座上演示自己的知识点。
#
# 说明：向量按内容哈希缓存在 .cache/emb.npz（首次真调、之后复用，避免反复花 token）。
# 没配 DASHSCOPE_API_KEY 时仍可用：向量直接从缓存读（是此前真实调用的结果），
# 但需要现场调用模型的重排/生成会打印录制结果并提示配置方式。
from dotenv import load_dotenv; load_dotenv()
import os, re, json, time, hashlib
from pathlib import Path
import numpy as np

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY
_DATA = Path('data') if Path('data').is_dir() else Path.cwd() / 'data'
_CACHE_FILE = Path('.cache') / 'emb.npz'
EMBED_MODEL = 'text-embedding-v3'
RERANK_MODEL = 'qwen3-rerank'
NO_KEY_TIP = ('未配置 DASHSCOPE_API_KEY：需要现场调用模型的部分将展示此前真实调用的录制结果，'
              '在项目根 .env 配置后自动变为实时调用。')

def recorded(text, note=''):
    """无 Key 时展示「此前真实运行的录制结果」。内容来自真实调用，不是编造的假数据。"""
    print(NO_KEY_TIP)
    print('—— 录制结果%s ——' % ('（' + note + '）' if note else ''))
    print(text)

if not _HAS_KEY:
    print(NO_KEY_TIP)

# ---------- 1) 语料：读 data/ 全部 Markdown，按小节切块 ----------
# 注意：评测集*.md 是「人工标注的答案」，不能进索引 —— 否则第 34 课评测时，
# 标注本身会被检索命中，指标虚高（数据泄漏）。这里按文件名前缀排除（含第 33 课产出的 评测集_v2.md）。
_EXCLUDE_PREFIX = '评测集'

def load_chunks(chunk_size=300, overlap=60):
    """按「## 小节」切分，小节过长再按句子窗口滑切。返回 [{'i','text','source','section'}]"""
    out = []
    for p in sorted(_DATA.glob('*.md')):
        if p.name.startswith(_EXCLUDE_PREFIX):
            continue
        section, buf = p.stem, []
        for line in p.read_text(encoding='utf-8').splitlines():
            if line.startswith('## '):
                if buf: out += _split_section(buf, section, p.name, chunk_size, overlap)
                section, buf = line[3:].strip(), [line]
            elif line.startswith('# '):
                section = line[2:].strip()
            else:
                buf.append(line)
        if buf: out += _split_section(buf, section, p.name, chunk_size, overlap)
    for i, c in enumerate(out):
        c['i'] = i
    return out

def _split_section(lines, section, source, chunk_size, overlap):
    """小节内容按句号聚合成 ~chunk_size 字的片段，相邻片段留 overlap 字重叠"""
    text = '\n'.join(lines).strip()
    if not text: return []
    sents = [s for s in re.split(r'(?<=[。！？\n])', text) if s.strip()]
    chunks, buf = [], ''
    for s in sents:
        if len(buf) + len(s) > chunk_size and buf:
            chunks.append(buf.strip())
            buf = buf[-overlap:] + s          # 保留尾部 overlap 字做上下文重叠
        else:
            buf += s
    if buf.strip(): chunks.append(buf.strip())
    return [{'text': c, 'source': source, 'section': section} for c in chunks]

# ---------- 2) 向量：真调 text-embedding-v3（分批 + 重试 + 内容哈希缓存）----------
def _load_cache():
    if not _CACHE_FILE.exists():
        return {}
    try:
        z = np.load(_CACHE_FILE, allow_pickle=False)
        return dict(zip(z['hashes'].tolist(), z['vectors']))
    except Exception as e:                      # 文件损坏（例如多进程同时写）：当空缓存重建，别让 notebook 挂掉
        print('向量缓存不可读(%s: %s)，将重新向量化：%s' % (type(e).__name__, e, _CACHE_FILE))
        return {}

def _save_cache(cache):
    """写盘前先与磁盘上已有内容合并，再原子替换 —— 避免多个进程同时跑时互相覆盖 / 写坏文件"""
    _CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
    for k, v in _load_cache().items():
        cache.setdefault(k, v)
    hs = np.array(list(cache.keys()))
    vs = np.array([cache[h] for h in cache.keys()], dtype='float32')
    # 进程号唯一，别抢同一个临时文件；注意 np.savez_compressed 会自动补 .npz 后缀，临时名必须也是 .npz 结尾
    tmp = _CACHE_FILE.with_name('%s.%d.tmp.npz' % (_CACHE_FILE.stem, os.getpid()))
    np.savez_compressed(tmp, hashes=hs, vectors=vs)
    try:
        os.replace(tmp, _CACHE_FILE)            # 原子替换：别的进程读到的永远是完整文件
    except OSError:                             # 目标被占用时稍等再试
        time.sleep(0.2); os.replace(tmp, _CACHE_FILE)

def _key(text, model):
    return hashlib.sha1((model + '\x00' + text).encode('utf-8')).hexdigest()[:16]

def embed(texts, model=EMBED_MODEL, batch=10):
    """真调 Embedding；命中缓存则直接用（缓存来自真实调用）。返回已 L2 归一化的向量"""
    if isinstance(texts, str): texts = [texts]
    cache, todo = _load_cache(), []
    for t in texts:
        k = _key(t, model)
        if k not in cache and k not in [x[0] for x in todo]:
            todo.append((k, t))
    if todo and not _HAS_KEY:
        raise RuntimeError('本地缓存缺少 %d 条向量，且未配置 DASHSCOPE_API_KEY，无法现场向量化。'
                           '请在项目根 .env 配置 Key 后重跑。' % len(todo))
    if todo:
        from dashscope import TextEmbedding
        pending = todo
        while pending:                                  # 批次过大就减半重试
            b = pending[:batch]
            r = TextEmbedding.call(model=model, input=[t for _, t in b], api_key=_KEY)
            if r.status_code == 200:
                for (k, _), e in zip(b, sorted(r.output['embeddings'], key=lambda e: e['text_index'])):
                    cache[k] = np.array(e['embedding'], dtype='float32')
                pending = pending[len(b):]
            elif batch > 1:
                batch //= 2
            else:
                raise RuntimeError('向量化失败: %s %s' % (r.code, r.message))
        _save_cache(cache)
    v = np.array([cache[_key(t, model)] for t in texts], dtype='float32')
    return v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-10)

# ---------- 3) 索引：FAISS（归一化后内积=余弦）+ BM25 ----------
import faiss
from rank_bm25 import BM25Okapi

def tokenize(text):
    """中文用「单字 + 相邻双字」切词，无需外部分词器（与第 16 课一致）"""
    t = re.sub(r'\s+', '', text)
    return [t[i] for i in range(len(t))] + [t[i:i + 2] for i in range(len(t) - 1)]

CHUNKS = load_chunks()
VECS = embed([c['text'] for c in CHUNKS])
INDEX = faiss.IndexFlatIP(VECS.shape[1]); INDEX.add(VECS)
BM25 = BM25Okapi([tokenize(c['text']) for c in CHUNKS])
print('语料就绪：%d 篇文档 → %d 个片段，向量维度 %d' % (len({c['source'] for c in CHUNKS}), len(CHUNKS), VECS.shape[1]))

# ---------- 4) 检索：稠密 / 稀疏 / 混合（RRF 融合）----------
def dense_retrieve(query, k=5):
    sims, ids = INDEX.search(embed(query), k)
    return [dict(CHUNKS[i], score=float(s), from_='dense') for i, s in zip(ids[0], sims[0]) if i != -1]

def sparse_retrieve(query, k=5):
    scores = BM25.get_scores(tokenize(query))
    top = np.argsort(-scores)[:k]
    return [dict(CHUNKS[i], score=float(scores[i]), from_='bm25') for i in top if scores[i] > 0]

def hybrid_retrieve(query, k=5, rrf_k=60, pool=10):
    """RRF 融合：score = Σ 1/(rrf_k + rank)，只用名次不用原始分数，天然可比"""
    fused = {}
    for name, hits in (('dense', dense_retrieve(query, pool)), ('bm25', sparse_retrieve(query, pool))):
        for rank, h in enumerate(hits, 1):
            cur = fused.setdefault(h['i'], dict(h, score=0.0, from_=set()))
            cur['score'] += 1.0 / (rrf_k + rank)
            cur['from_'].add(name)
    return sorted(fused.values(), key=lambda x: -x['score'])[:k]

# ---------- 5) 重排：真调 DashScope TextReRank ----------
def rerank(query, docs, top_n=3, model=RERANK_MODEL):
    """docs 可以是字符串列表或检索结果 dict 列表；返回 [(文档, 相关性分数)]"""
    texts = [d['text'] if isinstance(d, dict) else d for d in docs]
    if not texts: return []
    if not _HAS_KEY:
        print(NO_KEY_TIP); return [(t, None) for t in texts[:top_n]]
    from dashscope import TextReRank
    r = TextReRank.call(model=model, query=query, documents=texts,
                        top_n=min(top_n, len(texts)), return_documents=False, api_key=_KEY)
    if r.status_code != 200:
        raise RuntimeError('重排失败: %s %s' % (r.code, r.message))
    return [(texts[it['index']], float(it['relevance_score'])) for it in r.output['results']]

# ---------- 6) 生成：qwen-plus（带重试）+ 结构化 JSON 输出 ----------
def chat(prompt, system='你是严谨的 RAG 助手：只依据给定资料回答，资料里没有的就直说不知道。',
         temperature=0.3, model='qwen-plus', retries=3):
    if not _HAS_KEY:
        return None
    from dashscope import Generation
    for attempt in range(retries):
        r = Generation.call(model=model, messages=[{'role': 'system', 'content': system},
                                                   {'role': 'user', 'content': prompt}],
                            temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            return r.output.choices[0].message.content
        if attempt == retries - 1:
            raise RuntimeError('生成失败: %s %s' % (r.code, r.message))
        time.sleep(1.5 * (attempt + 1))          # 限流类错误退避重试
    return None

def chat_json(prompt, system='只输出 JSON，不要任何解释或代码块标记。', retries=2, **kw):
    """要求模型输出 JSON 并解析；解析失败时把报错回喂再试一次"""
    for attempt in range(retries + 1):
        out = chat(prompt, system=system, **kw)
        if out is None: return None
        seg = out[out.find('{'): out.rfind('}') + 1]     # 容忍 ```json 包裹与前后废话
        try:
            return json.loads(seg)
        except Exception as e:
            if attempt == retries: raise
            prompt = prompt + '\n\n上次输出无法解析(%s)，请只输出合法 JSON。' % e
    return None


In [ ]:
# 语义缓存(Semantic Cache)：相似问题命中缓存，跳过 LLM
# 与精确缓存的区别：精确缓存只认「问题原文完全相同」（key=问题哈希），
# 换个说法就失效；语义缓存把问题向量化，按余弦相似度找「历史上问过的等价问题」，
# 因此「怎么收费」与「计费方式是怎样的」能共用一份答案。
import time
import numpy as np

class SemanticCache:
    def __init__(self, thr=0.78):
        self.items, self.thr = [], thr          # (归一化向量, 答案, 原问题)

    def embed(self, s):
        """真调底座 embed()：text-embedding-v3，1024 维，已 L2 归一化（内积即余弦）。
           旧版这里用 ord(c)%16 的字符哈希拼 16 维向量，那是假向量，跟语义无关。"""
        return embed([s])[0]

    def best(self, q):
        """返回 (与缓存中最相似条目的余弦, 该条目)；空缓存返回 (0.0, None)"""
        e = self.embed(q)
        return max(((float(e @ i[0]), i) for i in self.items), default=(0.0, None))

    def get(self, q):
        s, it = self.best(q)
        return it[1] if it is not None and s >= self.thr else None

    def put(self, q, a):
        self.items.append((self.embed(q), a, q))

def rag_answer(q, k=3):
    """真实 RAG 链路：混合检索取片段 → qwen-plus 依据片段作答（缓存要缓存的正是这条链路的结果）"""
    hits = hybrid_retrieve(q, k=k)
    ctx = '\n\n'.join('[%s·%s] %s' % (h['source'], h['section'], h['text']) for h in hits)
    ans = chat('只依据下面资料回答问题，资料没写的就说不知道。\n'
               '<context>\n%s\n</context>\n问题：%s' % (ctx, q))
    return ans, hits

cache = SemanticCache()
Q1 = '星云客服机器人怎么收费？'

# ① 首次提问：缓存未命中 → 走完整链路（检索 + 生成），把答案写进缓存
if _HAS_KEY:
    t0 = time.time()
    ans1, hits1 = rag_answer(Q1)
    gen_ms = (time.time() - t0) * 1000
    print('① 首次提问「%s」——缓存未命中，真调检索+生成（%.0f ms）' % (Q1, gen_ms))
    print('   命中片段: %s' % ' | '.join('%s·%s' % (h['source'], h['section']) for h in hits1))
    print('   生成答案: %s' % ans1.replace('\n', ' ')[:150])
    cache.put(Q1, ans1)
else:
    recorded("""① 首次提问「星云客服机器人怎么收费？」——缓存未命中，真调检索+生成（2542 ms）
   命中片段: 星云客服FAQ.md·计费相关 | 计费与SLA.md·计费与 SLA 说明 | 星云智能产品手册.md·星云智能客服机器人产品手册
   生成答案: 按版本订阅，基础版 298 元/月、标准版 998 元/月、专业版 2980 元/月，企业版按需报价。公有云版本按成功返回的对话轮次计费，失败请求不计费。""",
             '录制于 2026-09-12，模型 qwen-plus；下面②③的向量与相似度仍来自真实缓存向量，可照常复现')
    cache.put(Q1, '按版本订阅，基础版 298 元/月、标准版 998 元/月、专业版 2980 元/月，企业版按需报价。'
                  '公有云版本按成功返回的对话轮次计费，失败请求不计费。')

# ② 换个说法的同一个问题 → 语义命中，直接拿缓存答案，不再调 LLM
print()
print('② 后续提问的命中判定（余弦 ≥ %.2f 即命中）：' % cache.thr)
for q, tag in [('星云智能客服的计费方式是怎样的？', '换个说法的同一问题'),
               ('今天天气怎么样？', '完全无关的问题'),
               ('这个客服机器人一个月要花多少钱', '口语化问法')]:
    s, it = cache.best(q)
    hit = cache.get(q)
    print('   问「%s」（%s）' % (q, tag))
    print('     最相似缓存条目: 「%s」  余弦=%.4f  → %s'
          % (it[2], s, '命中，直接返回缓存答案' if hit else '未命中，需走完整链路'))
    if hit:
        print('     缓存答案: %s' % hit.replace('\n', ' ')[:110])

# ③ 衡量收益：命中只花一次 embedding，检索与生成全省掉
t0 = time.time(); cache.get('星云智能客服的计费方式是怎样的？'); hit_ms = (time.time() - t0) * 1000
print()
print('③ 命中耗时 %.1f ms' % hit_ms)
print('   （这里快是因为问题向量本身也命中底座缓存；线上真实开销是「一次 embedding API ≈ 数十 ms」，'
      '仍远小于整条 RAG 链路）')
print('→ 语义缓存用「一次 embedding + 一次余弦比较」换掉整条 RAG 链路；'
      '阈值是关键旋钮：调高省得稳但会漏命中（换说法问不到老答案），调低命中多但可能答非所问。')

## 2. 其它优化清单

### 检索侧
- 向量维度/量化（PQ/标量量化）降内存与延迟；
- 用 HNSW/ANN 代替暴力检索；适度减小候选集再重排；
- 只对必要字段做 embedding，控制索引体积。

### 生成侧（往往收益最大）
- 精简上下文（重排取 top-k 而非全塞）；
- 精简 prompt、限制生成长度、用更快的模型做简单任务；
- 流式输出能先给用户首字，改善 TTFT 感知。

### 系统侧
- 精确缓存：同一问题原文命中，key 是问题哈希；
- 语义缓存：向量相似即命中，见上代码；
- 预加载索引/模型（消除 Cold Start）、按需缩放副本。

## 小结

- 延迟大头是 LLM 生成，其次是 embedding 和检索；
- 缓存先做精确缓存（零成本），再做语义缓存；
- 优化顺序建议：切上下文 → 缓存 → 索引加速 → 模型/流式。